# PyTorch + Training Basics

[PyTorch](https://pytorch.org/) is the open-source deep-learning library we will use in this tutorial session. Other widely used libraries include [TensorFlow](https://www.tensorflow.org/), [JAX](https://docs.jax.dev/en/latest/quickstart.html), [MindSpore](https://www.mindspore.cn/en/), and others.

![Framework comparison](https://softwaremill.com/user/pages/blog/229.ml-engineer-comparison-of-pytorch-tensorflow-jax-and-flax/image2.png?g-2f9bce85)

*Trends of paper implementations grouped by framework: comparison of PyTorch vs. TensorFlow ([viso.ai](https://viso.ai/deep-learning/pytorch-vs-tensorflow/)).*

These libraries share the same broad ideas: tensors, data-loading tools, model definitions, automatic differentiation, and optimizers. PyTorch is a good teaching tool because it keeps the mathematical objects visible. Tensors look like arrays, models are Python classes, and gradients are computed automatically through the operations you write. It is also widely used in research because it has a strong GPU ecosystem, a large library of standard neural-network components, and a workflow that scales from small notebooks to large training jobs.

For this tutorial, PyTorch gives us a useful compromise: we can see the mechanics of training without deriving and coding every derivative by hand. The goal is not to memorize every PyTorch feature. The goal is to understand the training pattern that will reappear in the image-classification notebooks:

1. put data in tensors,
2. define a model,
3. compute a loss,
4. backpropagate gradients,
5. update parameters with an optimizer.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Reproducibility makes examples easier to discuss.
torch.manual_seed(123)
np.random.seed(123)

plt.rcParams['figure.figsize'] = (5, 4)
plt.rcParams['axes.grid'] = True

## 1. Tensors

A tensor is the basic data container in PyTorch. In practice, tensors are multi-dimensional arrays with extra machinery for GPUs and automatic differentiation.

The most important things to track are:

- **shape**: how many examples, channels, pixels, or features are present,
- **dtype**: integer labels, floating-point images, etc.,
- **device**: CPU or GPU.

In [ ]:
# Tensor initialization (very similar to numpy!)
x = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
])

# Print its content and properties
print(x)
print('shape :', x.shape)
print('dtype :', x.dtype)
print('device:', x.device)

If you have a numpy array, can you convert it to torch?

Absolutely:

In [ ]:
# Initialize the numpy array with the same values
x = np.array([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0],
], dtype=np.float32)

# Convert the numpy array to a torch tensor
x = torch.from_numpy(x)
print(x)

What about the other way around?

In [ ]:
# Want to make a guess as to how to convert a numpy array to a torch tensor? Try it out!
# ...

### Prompt

Before running the next cell: what shape do you expect for `x[:, 1]`? Is it a column with shape `(3, 1)` or a flat tensor with shape `(3,)`?

In [ ]:
# Indexing and reshaping are the two tensor operations we will use constantly.
print('first row:', x[0])
print('second column:', x[:, 1])
print('flattened:', x.reshape(-1))
print('mean of each column:', x.mean(dim=0))

### Exercise A: Tensor Shapes

These are quick checks, not a long exercise. Fill them in if time allows.

In [ ]:
# Exercise A
# 1. Print the shape of the first row of x.
# 2. Reshape x into shape (1, 6).
# 3. Compute the sum of each row.

# Your code here.

In [ ]:
# Use a GPU if one is available. Everything below also runs on CPU.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

x = x.to(device)
print(x.device)

### GPU Acceleration

One reason PyTorch is useful is that the same tensor code can run on a GPU. GPUs are good at doing many simple arithmetic operations in parallel, which is exactly what large tensor operations and neural networks need.

There is a catch: moving data to the GPU has overhead. For a tiny calculation, the transfer can cost more than the computation. For a large calculation, the GPU can be dramatically faster.

In [ ]:
import time

# Function which times the execution of an expensive mathematical operation on the input tensor.
def timed_matrix_power_sum(tensor):
    start = time.perf_counter()
    value = (tensor ** 5).sum()
    if tensor.is_cuda:
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return value, elapsed

# Initialize a large random matrix (2500x2500) on the CPU and time the operation.
matrix_size = 2500
large_cpu = torch.randn(matrix_size, matrix_size)

_, cpu_time = timed_matrix_power_sum(large_cpu)
print(f'CPU compute time: {cpu_time:.4f} s')

# If a GPU is available, transfer the matrix to the GPU and time both the transfer and the compute time.
if torch.cuda.is_available():
    start = time.perf_counter()
    large_gpu = large_cpu.to('cuda')
    torch.cuda.synchronize()
    transfer_time = time.perf_counter() - start

    _, gpu_time = timed_matrix_power_sum(large_gpu)
    print(f'CPU -> GPU transfer time: {transfer_time:.4f} s')
    print(f'GPU compute time        : {gpu_time:.4f} s')
    print(f'Compute speed-up        : {cpu_time / gpu_time:.1f}x')
else:
    print('CUDA is not available here, so this cell only reports CPU timing.')

### Prompt

If moving the tensor to the GPU takes time, when is it worth doing?

Rule of thumb: move data to the GPU when the computation is expensive enough, and avoid moving tensors back and forth inside tight loops.

## 2. A Toy Classification Problem

We will classify red and blue points in two dimensions. This is deliberately tiny so we can see what the model is doing.

Each point has two input features. Each label is either 0 or 1.

In [ ]:
# Function which generates two blobs of 2D data points, one centered at (-1.5, -1.0) and the other at (1.5, 1.0).
def make_blobs(n_per_class=500):
    # Random locations of the points in each blob, with some noise.
    blue = torch.randn(n_per_class, 2) + torch.tensor([-1.5, -1.0])
    red = torch.randn(n_per_class, 2) + torch.tensor([1.5, 1.0])

    # Blob labels (0 for blue, 1 for red).
    data = torch.cat([blue, red], dim=0).float()
    label = torch.cat([
        torch.zeros(n_per_class),
        torch.ones(n_per_class),
    ]).float().view(-1, 1)

    return data, label


# Generate the blobs and print their shapes.
data, label = make_blobs()
print('data shape :', data.shape)
print('label shape:', label.shape)

In [ ]:
# Use matplotlib to visualize the blobs, coloring the points according to their labels.
def plot_points(data, label, ax=None):
    if ax is None:
        _, ax = plt.subplots()
    data_np = data.detach().cpu().numpy()
    label_np = label.detach().cpu().view(-1).numpy()
    ax.scatter(data_np[label_np == 0, 0], data_np[label_np == 0, 1], s=8, c='tab:blue', alpha=0.6)
    ax.scatter(data_np[label_np == 1, 0], data_np[label_np == 1, 1], s=8, c='tab:red', alpha=0.6)
    ax.set_xlabel('feature 0')
    ax.set_ylabel('feature 1')
    ax.set_aspect('equal', adjustable='box')
    return ax


plot_points(data, label);

### Prompt

What kind of boundary should separate these points: a straight line, a curve, or something more complicated?

This question matters because the model we choose determines the kinds of boundaries it can draw.

## 3. Model, Loss, Optimizer

A model is a function with trainable parameters. Here the model is one linear layer:

$$ f(x) = w_0 x_0 + w_1 x_1 + b $$

The model outputs one number called a **logit**. Large positive logits mean class 1. Large negative logits mean class 0.

We need three ingredients:

- a model, `torch.nn.Module`,
- a loss function, which says how wrong the model is,
- an optimizer, which updates the parameters.

### Prompt

If our input has 2 features and our output is 1 logit, what should the linear layer look like?

```python
self.linear = torch.nn.Linear(?, ?)
```

In [ ]:
# Fill-together version. This cell is intentionally not needed by later cells.
class LinearClassifierFillIn(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # self.linear = torch.nn.Linear(?, ?)

    def forward(self, x):
        # return self.linear(x)
        pass

In [ ]:
class LinearClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(2, 1)

    def forward(self, x):
        return self.linear(x)


model = LinearClassifier().to(device)
loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

print(model)

Before training, run all points through the model. The loss is large because the parameters are still random.

In [ ]:
# Move data and label to the same device as the model first! (check what happens if you don't)
data = data.to(device)
label = label.to(device)

# Forward pass: compute the model output and loss.
logits = model(data)
loss = loss_fn(logits, label)

print('logits shape:', logits.shape)
print('loss:', loss.item())

### Prompt

What do you expect after one optimizer step?

- Should the loss always go down?
- Should accuracy immediately become good?
- Which object changes: the data, the labels, or the model parameters?

One training step has three essential lines: clear old gradients, compute new gradients, then update the parameters.

In [ ]:
# Fill-together version. What are the three optimizer-step lines?

# optimizer.____()
# loss.____()
# optimizer.____()

In [ ]:
optimizer.zero_grad()
loss.backward()
optimizer.step()

new_loss = loss_fn(model(data), label)
print('loss before one step:', loss.item())
print('loss after one step :', new_loss.item())

## 4. Training Loop

Training repeats the same step many times. The loop below is intentionally short because the same structure will reappear in the MLP and CNN notebooks.

In [ ]:
# Given a set of logits, compute the accuracy of the model's predictions compared to the true labels.
def accuracy_from_logits(logits, label):
    prediction = (torch.sigmoid(logits) > 0.5).float()
    return (prediction == label).float().mean()

# Train the model for a number of steps, keeping track of the loss and accuracy at regular intervals.
def train(model, data, label, steps=300, lr=0.05):
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    # Loop over the training steps, performing forward and backward passes and updating the model parameters.
    for step in range(steps):
        logits = model(data)
        loss = loss_fn(logits, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 10 == 0 or step == steps - 1:
            acc = accuracy_from_logits(logits, label)
            history.append((step, loss.item(), acc.item()))

    return torch.tensor(history)


model = LinearClassifier().to(device)
history = train(model, data, label)

print('final loss    :', history[-1, 1].item())
print('final accuracy:', history[-1, 2].item())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(history[:, 0], history[:, 1])
ax[0].set_xlabel('step')
ax[0].set_ylabel('loss')

ax[1].plot(history[:, 0], history[:, 2])
ax[1].set_xlabel('step')
ax[1].set_ylabel('accuracy')
ax[1].set_ylim(0, 1.05);

In [ ]:
# Draw the decision boundary of the trained model on top of the data points.
def plot_decision_boundary(model, data, label, ax=None):
    if ax is None:
        _, ax = plt.subplots()

    # Create a mesh of points which covers the area of the data
    data_cpu = data.detach().cpu()
    x0 = torch.linspace(data_cpu[:, 0].min() - 0.5, data_cpu[:, 0].max() + 0.5, 150)
    x1 = torch.linspace(data_cpu[:, 1].min() - 0.5, data_cpu[:, 1].max() + 0.5, 150)
    xx0, xx1 = torch.meshgrid(x0, x1, indexing='ij')

    # Compute the sigmoid scores for every point in the mesh
    grid = torch.stack([xx0.reshape(-1), xx1.reshape(-1)], dim=1).to(next(model.parameters()).device)
    with torch.no_grad():
        score = torch.sigmoid(model(grid)).reshape(len(x0), len(x1)).cpu()

    # Draw 20 contours from the scores, from 0 to 1
    ax.contourf(xx0, xx1, score, levels=20, cmap='RdBu_r', alpha=0.35)

    # Draw one contour at the 0.5 score level, which corresponds to the decision boundary
    ax.contour(xx0, xx1, score, levels=[0.5], colors='black', linewidths=2)

    # Draw the data points
    plot_points(data_cpu, label.detach().cpu(), ax=ax)

    return ax


plot_decision_boundary(model, data, label);

### Exercise B: Training Knobs

Try one change at a time:

1. Change the learning rate from `0.05` to `0.005` or `0.5`.
2. Change the number of training steps.
3. Re-run the loss and accuracy plot.

Before running it, predict whether the model will train slower, faster, or become unstable.

In [ ]:
# Exercise B
# model = LinearClassifier().to(device)
# history = train(model, data, label, steps=..., lr=...)
# print(history[-1])

# Your code here.

## 5. Why Nonlinearity Matters

A linear model can only draw one straight decision boundary. Some data need more flexible boundaries.

In [ ]:
# Makes two sets of points on concentric rings, one centered at the origin with radius ~0.8 and
# the other with radius ~1.8, both with some noise.
def make_rings(n_per_class=700):
    # Angles and radii of the points in each ring, with some noise.
    angle0 = 2 * np.pi * torch.rand(n_per_class)
    angle1 = 2 * np.pi * torch.rand(n_per_class)

    radius0 = 0.8 + 0.12 * torch.randn(n_per_class)
    radius1 = 1.8 + 0.18 * torch.randn(n_per_class)

    # Compute the (x, y) coordinates of the points in each ring from polar coordinates.
    inner = torch.stack([radius0 * torch.cos(angle0), radius0 * torch.sin(angle0)], dim=1)
    outer = torch.stack([radius1 * torch.cos(angle1), radius1 * torch.sin(angle1)], dim=1)

    # Assign labels (0 for inner ring, 1 for outer ring) and combine the data into a single tensor.
    data = torch.cat([inner, outer], dim=0).float()
    label = torch.cat([torch.zeros(n_per_class), torch.ones(n_per_class)]).float().view(-1, 1)

    return data, label


ring_data, ring_label = make_rings()
ring_data = ring_data.to(device)
ring_label = ring_label.to(device)


In [ ]:
plot_points(ring_data, ring_label);

### Prompt

Can a straight line separate the inner ring from the outer ring? What accuracy would you expect from a linear model?

In [ ]:
linear_on_rings = LinearClassifier().to(device)
linear_history = train(linear_on_rings, ring_data, ring_label, steps=300, lr=0.05)

print('linear model accuracy:', linear_history[-1, 2].item())
plot_decision_boundary(linear_on_rings, ring_data, ring_label);

Stacking linear layers is not enough by itself: without an activation function, multiple linear transformations collapse into one linear transformation. The activation is what lets the model bend the decision boundary.

### Prompt

Where should the activation go: before the first linear layer, between linear layers, or only at the end?

In [ ]:
# Fill-together version. Where does the nonlinearity belong?
class SmallMLPFillIn(torch.nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(2, hidden),
            # torch.nn.____(),
            torch.nn.Linear(hidden, hidden),
            # torch.nn.____(),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class SmallMLP(torch.nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(2, hidden),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.LeakyReLU(),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.layers(x)


mlp = SmallMLP(hidden=16).to(device)
mlp_history = train(mlp, ring_data, ring_label, steps=800, lr=0.01)

print('MLP accuracy:', mlp_history[-1, 2].item())

In [ ]:
plot_decision_boundary(mlp, ring_data, ring_label);

## 6. Training vs. Validation

A model is optimized on the training set, so it can look better there than on independent data. This is why we keep a validation or test sample separate.

For this example, we make a deliberately harder dataset: two overlapping Gaussian populations. The classes are not perfectly separable, and we train a flexible model on only a small subset. This makes it possible for the model to do very well on the training points while doing noticeably worse on held-out points.

In [ ]:
# Make two blobs of data which are more overlapping than the previous blobs
def make_overlapping_blobs(n_per_class=700):
    blue = 1.05 * torch.randn(n_per_class, 2) + torch.tensor([-0.7, -0.35])
    red = 1.05 * torch.randn(n_per_class, 2) + torch.tensor([0.7, 0.35])

    data = torch.cat([blue, red], dim=0).float()
    label = torch.cat([torch.zeros(n_per_class), torch.ones(n_per_class)]).float().view(-1, 1)
    return data, label


# Record the loss/accuracy on an *independent* validation set during training, and return the history of these values.
def train_with_validation(model, train_data, train_label, val_data, val_label, steps=1500, lr=0.01):
    loss_fn = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for step in range(steps):
        train_logits = model(train_data)
        train_loss = loss_fn(train_logits, train_label)

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        if step % 20 == 0 or step == steps - 1:
            with torch.no_grad():
                val_logits = model(val_data)
                val_loss = loss_fn(val_logits, val_label)
                train_acc = accuracy_from_logits(train_logits, train_label)
                val_acc = accuracy_from_logits(val_logits, val_label)
            history.append((step, train_loss.item(), val_loss.item(), train_acc.item(), val_acc.item()))

    return torch.tensor(history)


# Generate an independent validation set and move it to the same device as the model.
validation_data, validation_label = make_overlapping_blobs()
validation_data = validation_data.to(device)
validation_label = validation_label.to(device)

# Use a tiny balanced training subset and keep the rest for validation.
n_per_class = len(validation_label) // 2
train_per_class = 50
rng = torch.Generator().manual_seed(321)
blue_train_indices = torch.randperm(n_per_class, generator=rng)[:train_per_class]
red_train_indices = n_per_class + torch.randperm(n_per_class, generator=rng)[:train_per_class]
train_indices = torch.cat([blue_train_indices, red_train_indices]).to(device)

all_indices = torch.arange(len(validation_label), device=device)
mask = torch.ones(len(validation_label), dtype=torch.bool, device=device)
mask[train_indices] = False
val_indices = all_indices[mask]

train_data = validation_data[train_indices]
train_label = validation_label[train_indices]
val_data = validation_data[val_indices]
val_label = validation_label[val_indices]

validation_model = SmallMLP(hidden=64).to(device)
validation_history = train_with_validation(
    validation_model,
    train_data,
    train_label,
    val_data,
    val_label,
    steps=1500,
    lr=0.01,
)

print('train accuracy:', validation_history[-1, 3].item())
print('val accuracy  :', validation_history[-1, 4].item())


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))

ax[0].plot(validation_history[:, 0], validation_history[:, 1], label='train')
ax[0].plot(validation_history[:, 0], validation_history[:, 2], label='validation')
ax[0].set_xlabel('step')
ax[0].set_ylabel('loss')
ax[0].legend()

ax[1].plot(validation_history[:, 0], validation_history[:, 3], label='train')
ax[1].plot(validation_history[:, 0], validation_history[:, 4], label='validation')
ax[1].set_xlabel('step')
ax[1].set_ylabel('accuracy')
ax[1].set_ylim(0, 1.05)
ax[1].legend();

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

plot_decision_boundary(validation_model, train_data, train_label, ax=ax[0])
ax[0].set_title('Boundary on training points')

plot_decision_boundary(validation_model, val_data, val_label, ax=ax[1])
ax[1].set_title('Same boundary on validation points')

fig.tight_layout();


### Prompt

Questions:
- If training accuracy is higher than validation accuracy, what are two possible explanations?
- How can we increase the agreement between train accuracy and validation accuracy?
  - What can we do on the model side?
  - What can we do on the dataset side?

### Exercise C: Capacity

Change `hidden=16` to a smaller or larger value. Predict what will happen before you run it.

Questions to keep in mind:

- Does a larger model always train better?
- Does a larger model always generalize better?
- How would this connect to image classification?

In [ ]:
# Exercise C
# mlp = SmallMLP(hidden=...).to(device)
# mlp_history = train(mlp, ring_data, ring_label, steps=800, lr=0.01)
# print('MLP accuracy:', mlp_history[-1, 2].item())
# plot_decision_boundary(mlp, ring_data, ring_label)

# Your code here.

## 7. The Pattern To Reuse

The image notebooks use the same pattern, just with larger tensors and richer models:

```python
for data, label in loader:
    logits = model(data)
    loss = loss_fn(logits, label)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

Everything else is architecture, data handling, and diagnostics.

The key message is that PyTorch handles the derivatives, but we are still responsible for choosing the data representation, model architecture, loss function, optimizer, and diagnostics.